# Modelado — IEEE-CIS Fraud Detection

Entrenamos y comparamos varios modelos sobre las variables creadas en `02_feature_engineering`, con validación temporal: entrenamiento con el periodo más antiguo y evaluación con el más reciente.

**Métrica principal: PR-AUC.** Con un 3,5% de fraude, la accuracy no es informativa (un modelo que nunca detecte fraude obtiene un 96,5%), y la ROC-AUC tiende a ser optimista con clases tan desbalanceadas. La PR-AUC se centra en la clase minoritaria; su referencia aleatoria es la tasa de fraude (≈0,035).

El umbral de decisión se elige al final, con **recall** (fraudes detectados) y **precisión** (alertas que son fraude real) como reflejo de los dos costes del negocio: fraudes aprobados y clientes legítimos bloqueados.

**Modelos**, de menor a mayor complejidad: baseline (`DummyClassifier`), regresión logística, Random Forest, LightGBM y XGBoost.

## Carga de datos

Cargamos los tres conjuntos generados en `02_feature_engineering`: entrenamiento, validación (el periodo posterior) y test (sin target).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Dummy
from sklearn.dummy import DummyClassifier

from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score, recall_score

#Regresion logistica
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

train = pd.read_parquet(PROCESSED_DIR / "train_fe.parquet")
valid = pd.read_parquet(PROCESSED_DIR / "valid_fe.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_fe.parquet")

print(train.shape, valid.shape, test.shape)

(472429, 541) (118108, 541) (506691, 540)


## Selección de columnas y separación del target

El split temporal ya se hizo en `02_feature_engineering`: `train_fe` contiene el 80% más antiguo de las transacciones etiquetadas y `valid_fe`, el 20% más reciente. La celda anterior comprueba que no hay solapamiento en el tiempo entre los tres conjuntos.

Excluimos dos columnas que no deben entrar al modelo:

- **`TransactionID`**: es un identificador único de cada transacción y no contiene información sobre el fraude. Lo guardamos aparte para el fichero de predicciones de test.
- **`TransactionDT`**: es el instante absoluto de la transacción. Como el test es posterior al entrenamiento, todos sus valores son mayores que cualquiera de los vistos al entrenar. El modelo aprendería reglas sobre momentos concretos del pasado que no se pueden aplicar a fechas nuevas. La información temporal que sí se repite ya está en `hour` y `weekday`.

Después separamos las variables (`X`) del target (`y`, `isFraud`) en cada conjunto. Test no tiene target: sus predicciones se evaluarán en Kaggle.

In [3]:
cat_cols = train.select_dtypes(include=["object", "string"]).columns
train[cat_cols].nunique().sort_values(ascending=False)

Series([], dtype: float64)

In [5]:
test_ids = test["TransactionID"]

train = train.drop(columns=["TransactionID", "TransactionDT"])
valid = valid.drop(columns=["TransactionID", "TransactionDT"])
test = test.drop(columns=["TransactionID", "TransactionDT"])

y_train = train["isFraud"]
y_valid = valid["isFraud"]

X_train = train.drop(columns="isFraud")
X_valid = valid.drop(columns="isFraud")
X_test = test

print(X_train.shape, X_valid.shape, X_test.shape)
print(f"Fraude en train: {y_train.mean():.3%} · en valid: {y_valid.mean():.3%}")

(472429, 538) (118108, 538) (506691, 538)
Fraude en train: 3.514% · en valid: 3.441%


## Baseline

Antes de entrenar ningún modelo, establecemos una referencia mínima con `DummyClassifier(strategy="prior")`. Este modelo no usa ninguna variable: asigna a todas las transacciones la misma probabilidad de fraude, igual a la proporción observada en entrenamiento, y por tanto siempre predice la clase mayoritaria (no fraude).

Cualquier modelo útil debe superarlo con claridad. Además, nos permite comprobar con números por qué la accuracy no sirve en este problema: un modelo que nunca detecta fraude obtiene una accuracy muy alta.

In [7]:
dummy = DummyClassifier(strategy="prior")
dummy.fit(X_train, y_train)

dummy_proba = dummy.predict_proba(X_valid)[:, 1]
dummy_pred = dummy.predict(X_valid)

print(f"PR-AUC:   {average_precision_score(y_valid, dummy_proba):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_valid, dummy_proba):.4f}")
print(f"Accuracy: {accuracy_score(y_valid, dummy_pred):.4f}")
print(f"Recall:   {recall_score(y_valid, dummy_pred):.4f}")

PR-AUC:   0.0344
ROC-AUC:  0.5000
Accuracy: 0.9656
Recall:   0.0000


El baseline asigna a todas las transacciones la misma probabilidad, la tasa media de fraude, así que no puede ordenarlas: su ROC-AUC es 0,5, el valor del azar, y su PR-AUC coincide con la tasa de fraude

## Regresión logística

Primer modelo con aprendizaje. Lo usamos como referencia interpretable: si los modelos de árboles la superan con claridad, la diferencia vendrá de relaciones no lineales e interacciones que un modelo lineal no puede captar.

El pipeline imputa los nulos con la mediana y escala las variables, dos pasos que la regresión logística necesita y los árboles no. Usamos `class_weight="balanced"` para compensar el desbalance de clases.

In [8]:
logreg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_valid)[:, 1]

print(f"PR-AUC:  {average_precision_score(y_valid, logreg_proba):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_valid, logreg_proba):.4f}")

NameError: name 'Pipeline' is not defined